# *** Create WB Diff function ***

# 0. Initial

## 0.0. Imports

In [1]:
import flopy as fp
import pandas as pd
from WS_Mdl.core.mdl import Mdl_N
from WS_Mdl.core.style import set_verbose
import numpy as np

from WS_Mdl.imod.msw.mete_grid import to_DF
from WS_Mdl.imod.msw.meteo import to_XA
from WS_Mdl.imod.prj import r_with_OBS
from WS_Mdl.xr.spatial import clip_Mdl_area as xr_clip_Mdl_area
import xarray as xra

import imod
from WS_Mdl.core.defaults import CRS
from WS_Mdl.core.style import style_reset, blue
from WS_Mdl.imod.ini import CeCes
from WS_Mdl.xr import diagnostics
from WS_Mdl.imod.pop.msw import load_Par_Out

## 0.1 Options

In [ ]:
set_verbose(False)
MdlN='NBr54'
MdlN_B='NBr52'
date='19961231'
cumulative=True
sum_Pkg=True
net_only = True
Pa_Out=None
# x_CeCes, y_CeCes = CeCes(MdlN)
set_verbose(True)

# 1. MF6 budget

In [ ]:
# Load basics
M = Mdl_N(MdlN)
M_B = Mdl_N(MdlN_B) if MdlN_B else Mdl_N(M.B)

In [143]:
start_date = pd.to_datetime( M.INI.SDATE, format='%Y%m%d')

In [144]:
# Load budget to dataframes. fp.utils.Mf6ListBudget returns a tuple. 1st item is WB for each SP. 2nd item is cumulative.
i = 1 if cumulative else 0
DF_1 = fp.utils.Mf6ListBudget(M.Pa.LST_Mdl).get_dataframes(start_datetime=start_date - pd.Timedelta(days=1), diff=net_only)[i]
DF_2 = fp.utils.Mf6ListBudget(M_B.Pa.LST_Mdl).get_dataframes(start_datetime=start_date - pd.Timedelta(days=1), diff=net_only)[i]

In [145]:
if date is None:
    date = min(DF_1.index[-1], DF_2.index[-1]).strftime('%Y-%m-%d') # Use latest common date if not specified

In [ ]:
S_1 = DF_1.loc[DF_1.index == date].squeeze()
S_2 = DF_2.loc[DF_2.index == date].squeeze()

In [147]:
S_1.index = S_1.index.str.upper()
S_2.index = S_2.index.str.upper()

In [148]:
DF = pd.DataFrame([S_1, S_2], index=[MdlN, MdlN_B]).T

In [149]:
DF.drop(index=['TOTAL_IN', 'TOTAL_OUT', 'total_in', 'total_out', 'percent_discrepancy', 'in-out', 'total', 'PERCENT_DISCREPANCY', 'IN-OUT', 'TOTAL'], inplace=True, errors='ignore')

In [150]:
if sum_Pkg:
    S_idx = DF.index.to_series().astype(str)
    S_suffix = np.where(
        S_idx.str.endswith('_IN'),
        '_IN',
        np.where(S_idx.str.endswith('_OUT'), '_OUT', '_NET'),
    )
    S_grp = S_idx.str[:3] + S_suffix
    DF = DF.groupby(S_grp, sort=False).sum(min_count=1)

In [151]:
if not net_only:
    # Add net rows
    S_i = DF.index.to_series()
    S_i_in = S_i[S_i.str.endswith('_IN')]
    S_i_out = S_i[S_i.str.endswith('_OUT')]
    S_i_net = S_i_in.str.replace('_IN', '_NET')
    DF_NET = DF.loc[S_i_in.values].set_axis(S_i_net.values, axis=0) - DF.loc[S_i_out.values].to_numpy()
    DF = pd.concat([DF, DF_NET], axis=0)

In [152]:
# Sort index
MF_indexes = [S_i_in, S_i_out, S_i_net] if not net_only else [DF.index.to_series()]
sorted_i = pd.concat( [i.sort_values() for i in  MF_indexes]).values
DF = DF.loc[sorted_i]

# 2. Load MSW budget

In [226]:
DF_MSW = pd.DataFrame()

In [227]:
for m in [M, M_B]:
    # Load MSW WB Out CSV and handle dates
    DF_MSW_m = pd.read_csv(m.Pa.MSW / 'msw/csv/tot_svat_dtgw.csv', index_col=False, skipinitialspace=True)
    DF_MSW_m['TimeStamp'] = pd.to_datetime(DF_MSW_m['TimeStamp'].str.replace(' 24:', ' 00:', regex=False)) # Convert TimeStamp to datetime, handling '24:' as '00:'

    # Calculate Sum for the same period as GW budget and append to DF
    date_selection = (DF_MSW_m['TimeStamp']>=start_date) & (DF_MSW_m['TimeStamp']<=pd.to_datetime(date))
    DF_MSW[m.MdlN] = DF_MSW_m.loc[date_selection].iloc[:, 3:].sum()

In [228]:
# DF_MSW = DF_MSW.rename(index={i:  and (len(i) == 10) and ('ic' not in i)}).groupby(level=0, sort=False).sum() # Sum decSNN to one row

In [229]:
DF_MSW.rename(index={i: i.replace('(m3)', '') for i in DF_MSW.index}, inplace=True)

In [230]:
DF_MSW_ = DF_MSW.copy()

In [231]:
d_MSW_Agg = {'decStot': [i for i in DF_MSW.index if ('decS' in i)],
             'Ps': ['Psgw', 'Pssw'],
             'Id': ['Idgw', 'Idsw', 'Iddgw', 'Iddsw'],
             'ETact': ['Esp', 'Eic', 'Epd', 'Ebs', 'Tact'],
             'qMF': ['qmodf', 'qsimcorrtot']} # , 'qsimtot']
for k in d_MSW_Agg:
    print(k, len(d_MSW_Agg[k]), '\n', d_MSW_Agg[k])
    DF_MSW.rename(index={i: k for i in d_MSW_Agg[k]}, inplace=True)
DF_MSW = DF_MSW.groupby(level=0, sort=False).sum()

decStot 21 
 ['decSic', 'decSpdmac', 'decSpdmic', 'decS01', 'decS02', 'decS03', 'decS04', 'decS05', 'decS06', 'decS07', 'decS08', 'decS09', 'decS10', 'decS11', 'decS12', 'decS13', 'decS14', 'decS15', 'decS16', 'decS17', 'decS18']
Ps 2 
 ['Psgw', 'Pssw']
Id 4 
 ['Idgw', 'Idsw', 'Iddgw', 'Iddsw']
ETact 5 
 ['Esp', 'Eic', 'Epd', 'Ebs', 'Tact']
qMF 2 
 ['qmodf', 'qsimcorrtot']


In [232]:
DF_MSW.drop(index=['qsimtot', 'vcr'], inplace=True)

# 3. Merge MSW & MF DFs

In [233]:
DF_ = pd.concat([DF, DF_MSW], axis=0)
DF_ = DF_.loc[(DF_!=0).any(axis=1)]

In [234]:
d_Par = {'Pm' : "Precipitation (measured)",
         'Ps' : "Sprinkling Precipitation",
         'ETact' : "Evapotranspiration (actual)",
         'qrun' : "Runoff",
         'decStot' : "total storage change",
         'qMF' : "MODFLOW Stress",
        #  'vcr' : "Water Balance Error",
        #  'qsimtot' : "SIMGRO stresses on GW",
         'RCH_NET' : "Recharge",
         'CHD_NET' : "Constant Head",
         'DRN_NET' : "Drainage",
         'RIV_NET' : "Rivers",
         'SFR_NET' : "Streams",
         'STO_NET' : "Storage",
         'WEL_NET' : "Wells",}
        #  'qsimcorrtot' : "convergence error"}

In [235]:
DF_['Parameter'] = DF_.index.map(d_Par)

In [236]:
MF_Par = [i for i in d_Par if '_NET' in i]
MSW_Par = [i for i in d_Par if '_NET' not in i]

In [237]:
DF_['Diff'] = DF_[MdlN] - DF_[MdlN_B]
DF_['Diff_%'] = DF_.apply(lambda x: x['Diff'] / x[MdlN_B] * 100 if pd.notnull(x['Diff']) and x[MdlN_B] != 0 else np.nan, axis=1)

In [238]:
DF_.insert(len(DF_.columns)-1, 'Parameter', DF_.pop('Parameter')) # Move Parameter to last column

In [239]:
DF_ = DF_.reindex(d_Par.keys())

In [208]:
DF_.style.format("{:,.0f}", subset=DF_.columns[:4])

,NBr54,NBr52,Diff,Diff_%,Parameter
Pm,"64,836,735","61,860,742","2,975,993",5,Precipitation (measured)
Ps,"2,926,633","2,749,410","177,223",6,Sprinkling Precipitation
ETact,"-46,991,120","-44,566,490","-2,424,630",5,Evapotranspiration (actual)
qrun,"-361,306","-3,339,785","2,978,479",-89,Runoff
decStot,"-1,220,318","1,459,169","-2,679,488",-184,total storage change
qMF,"-19,193,189","-18,166,141","-1,027,048",6,MODFLOW Stress
RCH_NET,"13,207,082","10,797,078","2,410,004",22,Recharge
CHD_NET,"-4,427,694","-4,497,534","69,840",-2,Constant Head
DRN_NET,"-837,279","-802,160","-35,119",4,Drainage
RIV_NET,"-2,814,093","-4,577,399","1,763,306",-39,Rivers


# 4. Test WB closing

In [246]:
DF_closing = pd.DataFrame()
DF_closing['MF SUM'] = DF_.loc[MF_Par, [MdlN, MdlN_B]].sum()
DF_closing['MSW SUM'] = DF_.loc[MSW_Par, [MdlN, MdlN_B]].sum()
DF_closing['MF ABS SUM'] = DF_.loc[MF_Par, [MdlN, MdlN_B]].apply(lambda x: x.abs()).sum()
DF_closing['MSW ABS SUM'] = DF_.loc[MSW_Par, [MdlN, MdlN_B]].apply(lambda x: x.abs()).sum()
DF_closing['MF SUM % error'] = DF_closing['MF SUM'] / DF_closing['MF ABS SUM'] * 100
DF_closing['MSW SUM % error'] = DF_closing['MSW SUM'] / DF_closing['MSW ABS SUM'] * 100
DF_closing.style.format("{:,.0f}", subset=DF_closing.columns[:4])

,MF SUM,MSW SUM,MF ABS SUM,MSW ABS SUM,MF SUM % error,MSW SUM % error
NBr54,"6,227","-2,565","38,366,981","135,529,301",0.016231,-0.001893
NBr52,737,"-3,095","36,622,488","132,141,737",0.002012,-0.002342


# -2. full MSW dict

In [ ]:
d_MSW_Par = {
    'decSic'      : {'Parameter': '-Δ interception', 'Description' : 'decrease of interception storage'},
    'decSpdmac'   : {'Parameter': '-Δ macro ponding', 'Description' : 'decrease of `macro` ponding storage'},
    'decSpdmic'   : {'Parameter': '-Δ micro ponding', 'Description' : 'decrease of `micro` ponding storage'},
    'decS_Tot'    : {'Parameter': '-Δ storage', 'Description' : 'decrease of water storage in rootzone boxes'},
    'Pm'          : {'Parameter': 'measured P', 'Description' : 'measured precipitation'},
    'Psgw'        : {'Parameter': 'sprinkling P - GW', 'Description' : 'sprinkling precipitation, from groundwater'},
    'Idgw'        : {'Parameter': 'drip irrigation - GW', 'Description' : 'drip irrigation, from groundwater'},
    'Iddgw'       : {'Parameter': 'deep drip irrigation - GW', 'Description' : 'deep drip irrigation, from groundwater'},
    'Pssw'        : {'Parameter': 'sprinkling P - SW', 'Description' : 'sprinkling precipitation, from surface water'},
    'Idsw'        : {'Parameter': 'drip irrigation - SW', 'Description' : 'drip irrigation, from surface water'},
    'Iddsw'       : {'Parameter': 'deep drip irrigation - SW', 'Description' : 'deep drip irrigation, from surface water'},
    'Esp'         : {'Parameter': 'E sprinkling', 'Description' : 'evaporation sprinkling water'},
    'Eic'         : {'Parameter': 'E interception', 'Description' : 'evaporation interception water'},
    'Epd'         : {'Parameter': 'E ponding', 'Description' : 'evaporation ponding water'},
    'Ebs'         : {'Parameter': 'E bare soil', 'Description' : 'evaporation bare soil'},
    'Tact'        : {'Parameter': 'T actual', 'Description' : 'actual transpiration vegetation'},
    'qrun'        : {'Parameter': 'runon', 'Description' : 'runon'},
    'qdr'         : {'Parameter': 'net infiltration of SW', 'Description' : 'net infltration of surface water'},
    'qdrrdi'      : {'Parameter': 'drainage blocking + subirrigation', 'Description' : 'drainage blocking plus subirrigation of RDI'},
    'qspgw'       : {'Parameter': 'GW for sprinkling', 'Description' : 'groundwater extraction for sprinkling from ly=1'},
    'qmodf'       : {'Parameter': 'MF stresses on GW', 'Description' : 'sum of all MODFLOW stresses on groundwater'},
    'vcr'         : {'Parameter': 'WB error', 'Description' : 'water balance error (water creation)'},
    'qsimtot'     : {'Parameter': 'SIMGRO stresses on GW', 'Description' : "sum of SIMGRO stresses on groundwater, including MODFLOW correction term"},
    'qsimcorrtot' : {'Parameter': 'convergence correction term', 'Description' : 'correction term for realignment of MF in the case that there was not a full convergence during the last TS'}
}

# -1. Junkyard - Old method of loading MSW IDF Outs (much slower than laoding the budget csv)

In [ ]:
# (DA_P*DA_P_A).sum(dim=['time', 'x', 'y']).values

In [ ]:
# (DA_AEVT*DA_AEVT_A).sum(dim=['time', 'x', 'y']).values

In [ ]:
# (DA_qrun*DA_qrun_A).sum(dim=['time', 'x', 'y']).values

## 2. Load P

In [ ]:
# DF_ = DF.copy()

In [ ]:
# DF_P_prev = None
# for m in [MdlN, MdlN_B]:
#     PRJ = r_with_OBS(M.Pa.PRJ)[0] # [0], cause [1] is the OBS

#     DF_P = to_DF(PRJ)

#     if not cumulative:
#         DF_P = DF_P.loc[DF_P['DT'] == pd.to_datetime(date)]
#     else:
#         DF_P = DF_P.loc[(DF_P['DT'] <= pd.to_datetime(date )) & (DF_P['DT'] >= start_date)]

#     if DF_P_prev is not None:
#         if DF_P['P'].equals(DF_P_prev):
#             print(f' -- {blue}{m}{style_reset}: Copied P from previous MdlN (identical).')
#             continue
#     else:
#         DF_P_prev = DF_P['P'].copy()

#     A_P = to_XA(DF_P, 'P', m)
#     A_P_refined = A_P.interp(coords={'x': x_CeCes, 'y': y_CeCes[::-1]}, method='nearest') # Refines and clips

#     DF.loc['P', m] = P = A_P_refined.sum(dim=['time', 'x', 'y']).values # asign to DF

## 3. Load AEVT

In [ ]:
# start_date = start_date.strftime('%Y%m%d')

In [ ]:
# DA_AEVT, DA_AEVT_A = load_Par_Out(MdlN, 'bdgETact', start_date, date) 

In [ ]:
# DA_P, DA_P_A = load_Par_Out(MdlN, 'bdgPm', start_date, date) 

## 3. Load Surface runoff

In [ ]:
# DA_qrun, DA_qrun_A = load_Par_Out(MdlN, 'bdgqrun', start_date, date) 